# Lora no Kaggle: quatro tamanhos de modelo na GPU T4

Treina o extrator de pedidos com o [Soup CLI](https://github.com/MakazhanAlpamys/Soup) em quatro tamanhos
(Qwen2.5 de 0,5B, 1,5B, 3B e 7B) e avalia todos no mesmo conjunto de teste, separando casos normais e casos de borda.

Antes de rodar, no painel lateral **Settings**:

1. **Accelerator**: `GPU T4 x2`
2. **Internet**: ligada (precisa de telefone verificado na conta)

Para rodar sem deixar o navegador aberto, use **Save Version → Save & Run All (Commit)**.
O total estimado é de 4 horas de GPU, mas essa estimativa não foi medida: confira o tempo do primeiro modelo.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi

## 2. Baixar o projeto e instalar o Soup

In [ ]:
# Pode rodar esta célula quantas vezes quiser: ela sempre volta para a pasta
# certa e atualiza o projeto, em vez de clonar de novo dentro dele.
%cd /kaggle/working
!rm -rf Lora/Lora
![ -d Lora/.git ] && git -C Lora pull -q || git clone -q https://github.com/FabioAGC/Lora.git
%cd /kaggle/working/Lora
!pip install -q "soup-cli[train]"


## 3. Treinar e avaliar

Usa uma só das duas T4. **Rode primeiro só o menor modelo** (cerca de 10 minutos): ele confirma que a GPU,
o Soup e a avaliação funcionam antes de você gastar horas. Depois que der certo, troque `MODELOS` por
`"0.5b 1.5b 3b 7b"` (ou só os que quiser) e rode de novo: modelos já avaliados são pulados.


In [ ]:
MODELOS = "0.5b"  # depois do primeiro teste: "0.5b 1.5b 3b 7b"
%env CUDA_VISIBLE_DEVICES=0
%env PYTHONPATH=src
!python -m pedido_lora.kaggle --modelos {MODELOS}


## 4. Resultado

In [ ]:
from IPython.display import Markdown
Markdown(open("results/tabela.md").read())

## 5. Testar uma mensagem com o maior modelo

In [ ]:
!python -m pedido_lora.inferir --modelo Qwen/Qwen2.5-7B-Instruct --quant4 --adaptador outputs/qwen_7b_qlora "me ve 2 xburguer sem cebola, nao pera, 3, e uma coquinha zero. quanto tempo demora a entrega?"

## 6. Guardar o resultado

Tudo que fica em `/kaggle/working` aparece na aba **Output** do notebook depois do commit, para baixar.
Os adaptadores LoRA são pequenos (dezenas de MB), então cabem com folga.

In [ ]:
!rm -rf /kaggle/working/resultado && mkdir -p /kaggle/working/resultado/adaptadores
!for d in outputs/qwen_*; do mkdir -p /kaggle/working/resultado/adaptadores/$(basename $d) && cp $d/adapter_* $d/*.json /kaggle/working/resultado/adaptadores/$(basename $d)/ 2>/dev/null; done
!cp -r results outputs/previsoes /kaggle/working/resultado/
!du -sh /kaggle/working/resultado/*

## 7. Publicar no Hugging Face (opcional)

Crie um token de escrita no Hugging Face e guarde no Kaggle em **Add-ons → Secrets** com o nome `HF_TOKEN`.
Troque `SEU_USUARIO` pelo seu usuário e `ADAPTADOR`/`BASE` pelo modelo que quiser publicar.

O model card (`docs/MODEL_CARD.md`) descreve a primeira versão do modelo: atualize o modelo base e a tabela
de resultados com os números da etapa 4 antes de publicar.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
ADAPTADOR, BASE = "outputs/qwen_7b_qlora", "Qwen/Qwen2.5-7B-Instruct"
!sed -i "s#base_model: .*#base_model: {BASE}#" docs/MODEL_CARD.md
!pip install -q huggingface_hub
!python -m pedido_lora.publicar --usuario SEU_USUARIO --adaptador {ADAPTADOR}